# 00 - Data Quality & Exploratory Data Analysis

This notebook performs initial data quality checks and exploratory analysis on the
Synthea-generated patient cohort. We examine:

- Patient demographics (age, sex distribution)
- Condition code frequencies (ICD-10 / SNOMED)
- Missingness patterns across clinical variables
- Temporal code drift (distribution of diagnosis codes over calendar time)
- Overall data quality summary and recommendations for downstream modelling

In [ ]:
# ── Imports and data loading ─────────────────────────────────────────────────
import pathlib
import warnings

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

DATA_DIR = pathlib.Path("../data/synthea")

patients = pd.read_csv(DATA_DIR / "patients.csv", parse_dates=["BIRTHDATE", "DEATHDATE"])
conditions = pd.read_csv(DATA_DIR / "conditions.csv", parse_dates=["START", "STOP"])
observations = pd.read_csv(DATA_DIR / "observations.csv", parse_dates=["DATE"])
medications = pd.read_csv(DATA_DIR / "medications.csv", parse_dates=["START", "STOP"])
encounters = pd.read_csv(DATA_DIR / "encounters.csv", parse_dates=["START", "STOP"])

print(f"Patients:     {len(patients):>8,}")
print(f"Conditions:   {len(conditions):>8,}")
print(f"Observations: {len(observations):>8,}")
print(f"Medications:  {len(medications):>8,}")
print(f"Encounters:   {len(encounters):>8,}")

In [ ]:
# ── Patient demographics summary ────────────────────────────────────────────
reference_date = pd.Timestamp("2024-01-01")
patients["age"] = (reference_date - patients["BIRTHDATE"]).dt.days / 365.25
patients["alive"] = patients["DEATHDATE"].isna()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age distribution
axes[0].hist(patients["age"].dropna(), bins=40, edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Count")
axes[0].set_title("Age distribution")
axes[0].axvline(patients["age"].median(), color="red", ls="--", label=f"Median: {patients['age'].median():.1f}")
axes[0].legend()

# Sex distribution
sex_counts = patients["GENDER"].value_counts()
axes[1].bar(sex_counts.index, sex_counts.values, color=["#4C72B0", "#DD8452"])
axes[1].set_title("Sex distribution")
axes[1].set_ylabel("Count")
for i, (label, val) in enumerate(sex_counts.items()):
    axes[1].text(i, val + 10, f"{val:,}", ha="center", fontsize=10)

# Alive vs deceased
alive_counts = patients["alive"].value_counts()
axes[2].bar(["Alive", "Deceased"], [alive_counts.get(True, 0), alive_counts.get(False, 0)],
            color=["#55A868", "#C44E52"])
axes[2].set_title("Vital status")
axes[2].set_ylabel("Count")

fig.suptitle("Patient Demographics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(patients[["age", "GENDER"]].describe(include="all"))

In [ ]:
# ── Condition code frequency analysis ───────────────────────────────────────
code_freq = conditions["DESCRIPTION"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 25 most frequent conditions
top_n = 25
code_freq.head(top_n).plot.barh(ax=axes[0], color="#4C72B0")
axes[0].set_xlabel("Frequency")
axes[0].set_title(f"Top {top_n} condition descriptions")
axes[0].invert_yaxis()

# Cumulative coverage
cumulative = code_freq.cumsum() / code_freq.sum()
axes[1].plot(range(1, len(cumulative) + 1), cumulative.values, lw=2)
axes[1].axhline(0.80, color="red", ls="--", label="80% coverage")
n_80 = (cumulative <= 0.80).sum() + 1
axes[1].axvline(n_80, color="orange", ls="--", label=f"{n_80} codes for 80%")
axes[1].set_xlabel("Number of unique codes")
axes[1].set_ylabel("Cumulative proportion")
axes[1].set_title("Cumulative code coverage")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Total unique condition codes: {conditions['CODE'].nunique()}")
print(f"Total unique descriptions:    {conditions['DESCRIPTION'].nunique()}")
print(f"Codes needed for 80% coverage: {n_80}")

In [ ]:
# ── Missingness heatmap ─────────────────────────────────────────────────────
# Merge key tables to build a wide patient-level feature frame
# and inspect missingness patterns.

# Build a simple patient-level feature table with common clinical fields
obs_wide = (
    observations
    .groupby(["PATIENT", "DESCRIPTION"])["VALUE"]
    .last()
    .unstack()
)

# Select commonly expected clinical observations
clinical_cols = [
    "Body Height", "Body Weight", "Body Mass Index",
    "Systolic Blood Pressure", "Diastolic Blood Pressure",
    "Heart rate", "Respiratory rate",
    "Hemoglobin A1c/Hemoglobin.total in Blood",
    "Glucose", "Total Cholesterol", "Low Density Lipoprotein Cholesterol",
    "High Density Lipoprotein Cholesterol", "Triglycerides",
    "Estimated Glomerular Filtration Rate",
    "Creatinine", "Calcium", "Potassium",
]
available_cols = [c for c in clinical_cols if c in obs_wide.columns]
obs_subset = obs_wide[available_cols]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Missingness bar chart
missing_pct = obs_subset.isna().mean().sort_values(ascending=False)
missing_pct.plot.barh(ax=axes[0], color="#C44E52")
axes[0].set_xlabel("Fraction missing")
axes[0].set_title("Missingness by clinical observation")
axes[0].axvline(0.5, color="black", ls="--", alpha=0.5)

# Missingness matrix (sample for readability)
msno.matrix(obs_subset.sample(min(500, len(obs_subset)), random_state=42),
            ax=axes[1], sparkline=False, fontsize=8)
axes[1].set_title("Missingness matrix (sample of 500 patients)")

plt.tight_layout()
plt.show()

print("\nMissingness summary:")
print(missing_pct.to_frame("pct_missing").to_string())

In [ ]:
# ── Code drift analysis (temporal distribution of diagnosis codes) ──────────
conditions["year"] = conditions["START"].dt.year

# Focus on CVD- and diabetes-related codes for the two main tracks
cvd_keywords = ["coronary", "myocardial", "stroke", "heart failure", "atrial fibrillation"]
dm_keywords = ["diabetes", "prediabetes", "hba1c", "chronic kidney", "retinopathy"]

def flag_track(desc: str) -> str:
    desc_lower = str(desc).lower()
    if any(kw in desc_lower for kw in cvd_keywords):
        return "CVD"
    if any(kw in desc_lower for kw in dm_keywords):
        return "Diabetes"
    return "Other"

conditions["track"] = conditions["DESCRIPTION"].apply(flag_track)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overall code volume by year
yearly = conditions.groupby("year").size()
yearly.plot(ax=axes[0], marker="o", lw=2)
axes[0].set_title("Total condition records per year")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("Year")

# Track-specific trends
track_yearly = conditions.groupby(["year", "track"]).size().unstack(fill_value=0)
for col in ["CVD", "Diabetes"]:
    if col in track_yearly.columns:
        axes[1].plot(track_yearly.index, track_yearly[col], marker="o", label=col, lw=2)
axes[1].set_title("CVD vs Diabetes condition records per year")
axes[1].set_ylabel("Count")
axes[1].set_xlabel("Year")
axes[1].legend()

plt.tight_layout()
plt.show()

# Check for sudden jumps that may indicate coding-practice changes
pct_change = yearly.pct_change().dropna()
outlier_years = pct_change[pct_change.abs() > 0.5]
if len(outlier_years) > 0:
    print("WARNING: Large year-over-year volume changes detected:")
    for yr, chg in outlier_years.items():
        print(f"  {yr}: {chg:+.1%}")
else:
    print("No major year-over-year volume jumps detected (threshold: 50%).")

In [ ]:
# ── Data quality summary ────────────────────────────────────────────────────
summary = {
    "Total patients": len(patients),
    "Alive patients": int(patients["alive"].sum()),
    "Deceased patients": int((~patients["alive"]).sum()),
    "Median age (years)": round(patients["age"].median(), 1),
    "Sex ratio (M:F)": f"{(patients['GENDER']=='M').sum()}:{(patients['GENDER']=='F').sum()}",
    "Total condition records": len(conditions),
    "Unique condition codes": conditions["CODE"].nunique(),
    "Date range (conditions)": f"{conditions['START'].min().date()} to {conditions['START'].max().date()}",
    "Total observation records": len(observations),
    "Total medication records": len(medications),
    "CVD-related condition records": int((conditions["track"] == "CVD").sum()),
    "Diabetes-related condition records": int((conditions["track"] == "Diabetes").sum()),
}

summary_df = pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])
print("=" * 55)
print("        DATA QUALITY SUMMARY")
print("=" * 55)
print(summary_df.to_string())
print("=" * 55)

# Recommendations
print("\nRecommendations:")
print("  1. Drop observations with >60% missingness before modelling.")
print("  2. Monitor code drift - consider time-stratified train/test splits.")
print("  3. Validate SNOMED-to-ICD-10 mapping before building codelists.")
print("  4. Check for duplicate encounters (same patient, same day, same code).")
print("  5. Consider imputation strategy for labs with moderate missingness (20-50%).")